In [2]:
!rm -rf /content/drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
PROJECT_PATH = "/content/drive/MyDrive/Welding-Defect-Detection"

DATASET_V1 = f"{PROJECT_PATH}/The Welding Defect Dataset - v1"
DATASET_V2 = f"{PROJECT_PATH}/The Welding Defect Dataset - v2"

In [5]:
!ls "/content/drive/MyDrive/Welding-Defect-Detection"

 models      requirements.txt  'The Welding Defect Dataset - v1'   webapp
 notebooks   scripts	       'The Welding Defect Dataset - v2'


In [6]:
!pip install torch torchvision opencv-python tqdm

In [7]:
import os
import torch
import torchvision
import cv2
import numpy as np

from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from tqdm import tqdm

In [8]:
BATCH_SIZE = 8
IMAGE_SIZE = 640
LEARNING_RATE = 5e-5

EPOCHS_V1 = 10
EPOCHS_V2 = 5

DATASET_V1 = f"{PROJECT_PATH}/The Welding Defect Dataset - v1"
DATASET_V2 = f"{PROJECT_PATH}/The Welding Defect Dataset - v2"

MODEL_DIR = f"{PROJECT_PATH}/models"

os.makedirs(MODEL_DIR, exist_ok=True)

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

torch.backends.cudnn.benchmark = True

Using device: cuda
GPU: Tesla T4


In [10]:
class WeldDataset(Dataset):

    def __init__(self, img_dir, label_dir):

        self.img_dir = img_dir
        self.label_dir = label_dir
        self.images = os.listdir(img_dir)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        img_name = self.images[idx]

        img_path = os.path.join(self.img_dir, img_name)
        label_path = os.path.join(self.label_dir, os.path.splitext(img_name)[0] + ".txt")

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        image = cv2.resize(image,(IMAGE_SIZE,IMAGE_SIZE))

        h,w = IMAGE_SIZE,IMAGE_SIZE

        boxes=[]
        labels=[]

        if os.path.exists(label_path):

            with open(label_path) as f:

                for line in f.readlines():

                    c,x,y,bw,bh = map(float,line.split())

                    xmin=max(0,(x-bw/2)*w)
                    ymin=max(0,(y-bh/2)*h)
                    xmax=min(w,(x+bw/2)*w)
                    ymax=min(h,(y+bh/2)*h)

                    if xmax<=xmin or ymax<=ymin:
                        continue

                    boxes.append([xmin,ymin,xmax,ymax])

                    labels.append(int(c)+1)

        if len(boxes)==0:

            boxes=torch.zeros((0,4),dtype=torch.float32)
            labels=torch.zeros((0,),dtype=torch.int64)

        else:

            boxes=torch.tensor(boxes,dtype=torch.float32)
            labels=torch.tensor(labels,dtype=torch.int64)

        target={}
        target["boxes"]=boxes
        target["labels"]=labels
        target["image_id"]=torch.tensor([idx])

        image=torch.tensor(image).permute(2,0,1).float()/255.0

        return image,target

In [11]:
def collate_fn(batch):
    return tuple(zip(*batch))


def get_loader(dataset_path):

    img_dir = os.path.join(dataset_path,"train","images")
    label_dir = os.path.join(dataset_path,"train","labels")

    dataset = WeldDataset(img_dir,label_dir)

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        collate_fn=collate_fn
    )

    return loader

In [12]:
num_classes = 4

model = torchvision.models.detection.retinanet_resnet50_fpn(
    weights="DEFAULT"
)

num_anchors = model.head.classification_head.num_anchors
in_channels = model.backbone.out_channels

model.head.classification_head = torchvision.models.detection.retinanet.RetinaNetClassificationHead(
    in_channels,
    num_anchors,
    num_classes
)

model.to(device)

RetinaNet(
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(inplace=True)
          (downsample): Sequential(
            (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
            (1): FrozenBatchNorm2d(256, eps=0.0)


In [13]:
optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4
)

In [14]:
def train_model(model, loader, optimizer, epochs):

    scaler = torch.cuda.amp.GradScaler()

    for epoch in range(epochs):

        model.train()

        running_loss = 0

        loop = tqdm(loader)

        for images,targets in loop:

            images=[img.to(device) for img in images]
            targets=[{k:v.to(device) for k,v in t.items()} for t in targets]

            with torch.cuda.amp.autocast():

                loss_dict=model(images,targets)
                losses=sum(loss for loss in loss_dict.values())

            if torch.isnan(losses):
                print("Skipping NaN batch")
                continue

            optimizer.zero_grad()

            scaler.scale(losses).backward()

            scaler.step(optimizer)

            scaler.update()

            running_loss+=losses.item()

            loop.set_postfix(loss=losses.item())

        print(f"Epoch {epoch+1} Loss: {running_loss/len(loader):.4f}")

In [15]:
print("Training on Dataset V1")

loader = get_loader(DATASET_V1)

train_model(model, loader, optimizer, EPOCHS_V1)

torch.save(model.state_dict(), os.path.join(MODEL_DIR,"weld_v1.pth"))

Training on Dataset V1


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/tmp/ipykernel_3312/506488760.py:3: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
  0%|          | 0/105 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aw

Epoch 1 Loss: 1.4053


100%|██████████| 105/105 [01:24<00:00,  1.25it/s, loss=0.976]


Epoch 2 Loss: 1.0132


...

Epoch 8 Loss: 0.4777


100%|██████████| 105/105 [01:22<00:00,  1.28it/s, loss=0.335]


Epoch 9 Loss: 0.4297


100%|██████████| 105/105 [01:21<00:00,  1.28it/s, loss=0.38]


Epoch 10 Loss: 0.3950


In [16]:
print("Training on Dataset V2")

loader = get_loader(DATASET_V2)

train_model(model, loader, optimizer, EPOCHS_V2)

torch.save(model.state_dict(), os.path.join(MODEL_DIR,"weld_final.pth"))

print("Training complete")

Training on Dataset V2


/tmp/ipykernel_3312/506488760.py:3: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
  0%|          | 0/201 [00:00<?, ?it/s]/tmp/ipykernel_3312/506488760.py:18: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
100%|██████████| 201/201 [04:47<00:00,  1.43s/it, loss=0.701]


Epoch 1 Loss: 0.6510


100%|██████████| 201/201 [02:38<00:00,  1.27it/s, loss=0.364]


Epoch 2 Loss: 0.4820


...

Epoch 3 Loss: 0.5099


100%|██████████| 201/201 [02:36<00:00,  1.28it/s, loss=0.559]


Epoch 4 Loss: 0.3670


100%|██████████| 201/201 [02:37<00:00,  1.28it/s, loss=0.242]


Epoch 5 Loss: 0.3217
Training complete


In [1]:
import torch
import torchvision

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_PATH = f"{PROJECT_PATH}/models/weld_final.pth"

num_classes = 4

model = torchvision.models.detection.retinanet_resnet50_fpn(weights=None)

num_anchors = model.head.classification_head.num_anchors
in_channels = model.backbone.out_channels

model.head.classification_head = torchvision.models.detection.retinanet.RetinaNetClassificationHead(
    in_channels,
    num_anchors,
    num_classes
)

model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device)
model.eval()

print("Model loaded successfully")

NameError: name 'PROJECT_PATH' is not defined

In [ ]:
from torch.utils.data import DataLoader

TEST_DATASET = DATASET_V2

img_dir = os.path.join(TEST_DATASET,"test","images")
label_dir = os.path.join(TEST_DATASET,"test","labels")

test_dataset = WeldDataset(img_dir,label_dir)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn
)

print("Test images:",len(test_dataset))

In [ ]:
from torchvision.ops import box_iou
from tqdm import tqdm

TP = 0
FP = 0
FN = 0

IOU_THRESHOLD = 0.5

with torch.no_grad():

    for images,targets in tqdm(test_loader):

        images=[img.to(device) for img in images]

        outputs=model(images)

        for pred,target in zip(outputs,targets):

            pred_boxes = pred["boxes"].cpu()
            pred_scores = pred["scores"].cpu()

            gt_boxes = target["boxes"]

            pred_boxes = pred_boxes[pred_scores > 0.5]

            if len(pred_boxes)==0 and len(gt_boxes)==0:
                continue

            if len(pred_boxes)==0:
                FN += len(gt_boxes)
                continue

            if len(gt_boxes)==0:
                FP += len(pred_boxes)
                continue

            ious = box_iou(pred_boxes, gt_boxes)

            max_iou,_ = ious.max(dim=1)

            TP += (max_iou > IOU_THRESHOLD).sum().item()
            FP += (max_iou <= IOU_THRESHOLD).sum().item()

In [ ]:
precision = TP / (TP + FP + 1e-6)
recall = TP / (TP + FN + 1e-6)
f1 = 2 * precision * recall / (precision + recall + 1e-6)

print("\nEvaluation Results")
print("----------------------")

print("True Positives:", TP)
print("False Positives:", FP)
print("False Negatives:", FN)

print("\nPrecision:", round(precision,4))
print("Recall:", round(recall,4))
print("F1 Score:", round(f1,4))